In [1]:
print("Hellow qworld")

Hellow qworld


In [2]:
import numpy as np

In [3]:
data=np.load("files/mnist.npz")

In [4]:
X_train=data['x_train']
y_train=data['y_train']
X_test=data['x_test']
y_test=data['y_test']

In [5]:
X_train.shape


(60000, 28, 28)

In [6]:
#we have to flatten the images...because it means each row has 60000 images, each image has 28 rows and each row has 28 columns...
#we flatten each 28*28 image into 1D vector

In [7]:
y_train.shape

(60000,)

In [8]:
#flatten
X_train=X_train.reshape(X_train.shape[0],28*28)
X_test=X_test.reshape(X_test.shape[0],28*28)



In [9]:
#Normalize the values...so that inputs are on similar scale 

X_train=X_train/255.0
X_test=X_test/255.0

In [10]:
# Convert labels to integer type first
y_train = y_train.astype(int)
y_test = y_test.astype(int)

#Encoding for labels..
def one_hot(y, num_classes=10):
    one_hot_y = np.zeros((y.shape[0], num_classes))
    #all entries are 0
    one_hot_y[np.arange(y.shape[0]), y] = 1
    #only one position is 1
    return one_hot_y

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

In [11]:
#Activation function: ReLu
def relu(Z):
    # Replace all negative values with 0
    A = np.maximum(0, Z)
    return A

# Derivative of ReLU
def relu_derivative(Z):
    # If Z > 0, derivative is 1, else 0
    dZ = np.where(Z > 0, 1, 0)
    return dZ

In [12]:
#softmax function, it converts the probabilties: 
def softmax(Z):
    # Step 1: subtract largest value in each row (for stability)
    max_values = np.max(Z, axis=1, keepdims=True)
    Z_shifted = Z - max_values

    # Step 2: take exponent of each value
    exp_Z = np.exp(Z_shifted)

    # Step 3: sum exponentials row-wise
    sum_exp_Z = np.sum(exp_Z, axis=1, keepdims=True)

    # Step 4: divide each exponent by row sum
    A = exp_Z / sum_exp_Z

    return A

In [16]:
#Cross-Entropy Loss and accuracy: 
def compute_loss(Y_true, Y_pred):
    # Number of training examples
    m = Y_true.shape[0]

    # Small value added to avoid log(0)
    epsilon = 1e-8

    # Cross-entropy loss formula
    loss = -np.sum(Y_true * np.log(Y_pred + epsilon)) / m

    return loss

def compute_accuracy(X, y_true, W1, b1, W2, b2):
    # Get predictions from forward propagation
    Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)

    # Take the index of the largest probability in each row
    predictions = np.argmax(A2, axis=1)

    # Compare predictions with actual labels and compute accuracy
    accuracy = np.mean(predictions == y_true)
    #np.mean(....) treats True=1 and False=0
    return accuracy

In [14]:
#Initialize paramters: 
input_size = 784
hidden_size = 128
output_size = 10

np.random.seed(42)

# initialization for ReLU
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
#He inititalization formula...W1=random values x proper scaling 
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
b2 = np.zeros((1, output_size))

In [15]:
#Forward propagation: 
def forward_propagation(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

In [17]:
#Backpropagation
def back_propagation(X, Y, Z1, A1, A2, W2):
    m = X.shape[0]

    # Output layer gradients
    dZ2 = A2 - Y
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Hidden layer gradients
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2


In [23]:
#Traiining paramters: hyperparamters 
epochs = 45 #network sees the training dataset epoch times
learning_rate = 0.05 
batch_size = 100 #bumber of samples the model studies before making one learning correction, on before one back propogation....
#full-batch gradient descent is slower,memeory heavy,fewer updates 
num_samples = X_train.shape[0]

# to store loss history
loss_history = []

In [24]:
for epoch in range(epochs):
    # Shuffle training data
    permutation = np.random.permutation(num_samples)
    X_train_shuffled = X_train[permutation]
    Y_train_shuffled = Y_train[permutation]

    # Mini-batch gradient descent
    for i in range(0, num_samples, batch_size):
        X_batch = X_train_shuffled[i:i + batch_size]
        Y_batch = Y_train_shuffled[i:i + batch_size]

        # Forward pass
        Z1, A1, Z2, A2 = forward_propagation(X_batch, W1, b1, W2, b2)

        # Backward pass
        dW1, db1, dW2, db2 = back_propagation(X_batch, Y_batch, Z1, A1, A2, W2)

        # Parameter update
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    # Evaluate after each epoch
    Z1_train, A1_train, Z2_train, A2_train = forward_propagation(X_train, W1, b1, W2, b2)
    train_loss = compute_loss(Y_train, A2_train)
    train_acc = compute_accuracy(X_train, y_train, W1, b1, W2, b2)
    test_acc = compute_accuracy(X_test, y_test, W1, b1, W2, b2)

    loss_history.append(train_loss)
    
    #Print progress for this epoch
    print("Epoch:", epoch + 1, "/", epochs)
    print("Training Loss:", round(train_loss, 4))
    print("Training Accuracy:", round(train_acc, 4))
    print("Testing Accuracy:", round(test_acc, 4))
    print("-" * 50)


Epoch: 1 / 45
Training Loss: 0.0969
Training Accuracy: 0.9731
Testing Accuracy: 0.9685
--------------------------------------------------
Epoch: 2 / 45
Training Loss: 0.0908
Training Accuracy: 0.9752
Testing Accuracy: 0.969
--------------------------------------------------
Epoch: 3 / 45
Training Loss: 0.0858
Training Accuracy: 0.9768
Testing Accuracy: 0.9702
--------------------------------------------------
Epoch: 4 / 45
Training Loss: 0.0845
Training Accuracy: 0.9765
Testing Accuracy: 0.9701
--------------------------------------------------
Epoch: 5 / 45
Training Loss: 0.0792
Training Accuracy: 0.9786
Testing Accuracy: 0.9715
--------------------------------------------------
Epoch: 6 / 45
Training Loss: 0.0749
Training Accuracy: 0.9803
Testing Accuracy: 0.9734
--------------------------------------------------
Epoch: 7 / 45
Training Loss: 0.0721
Training Accuracy: 0.9808
Testing Accuracy: 0.9732
--------------------------------------------------
Epoch: 8 / 45
Training Loss: 0.0685

In [25]:
#Final evaluation

final_test_acc = compute_accuracy(X_test, y_test, W1, b1, W2, b2)
print(final_test_acc)

0.9786


In [ ]:
#for epoch=20, learning rate=0.01, and batch size=64..acurracy= 0.9546
#for epoch=45, lr=0.05,batch size=100...accuracy=0.9786
